In [0]:
from pyspark.sql.functions import col,sum,avg,floor,max,row_number,rank,dense_rank,explode,count
from pyspark.sql.window import Window
df_employee=spark.table("employee")
df_dept=spark.table("department")
df_customer=spark.table("customer")
df_order=spark.table("orders")   
df_invoice=spark.table("invoice")


In [0]:
#Get employee name with department name
df_employee.alias("e").join(df_dept.alias("d"),col("e.dept_id")==col("d.dept_id"),"inner").select("e.emp_name","d.dept_name").show()

+--------+---------+
|emp_name|dept_name|
+--------+---------+
|    Ravi|       HR|
|    Amit|       IT|
|   Sneha|       IT|
|   Kiran|  Finance|
|    Neha|    Sales|
|   Arjun|       IT|
+--------+---------+



In [0]:
#Get all employees including those without departments
df_employee.alias("e").join(df_dept.alias("d"), col("e.dept_id")==col("d.dept_id"),"left").select("e.emp_name","e.dept_id","d.dept_name").show()



+--------+-------+---------+
|emp_name|dept_id|dept_name|
+--------+-------+---------+
|    Ravi|    101|       HR|
|    Amit|    102|       IT|
|   Sneha|    102|       IT|
|   Kiran|    103|  Finance|
|    Neha|    104|    Sales|
|   Arjun|    102|       IT|
|  Aarush|    105|     NULL|
+--------+-------+---------+



In [0]:
# Find total salary department-wise
df_employee.groupBy("dept_id").agg(sum("salary")).show()

+-------+-----------+
|dept_id|sum(salary)|
+-------+-----------+
|    104|    55000.0|
|    103|    45000.0|
|    102|   215000.0|
|    101|    50000.0|
|    105|    50000.0|
+-------+-----------+



In [0]:
#Find average salary by department
df_employee.groupBy("dept_id").agg(floor(avg("salary"))).show()

+-------+------------------+
|dept_id|FLOOR(avg(salary))|
+-------+------------------+
|    104|             55000|
|    103|             45000|
|    102|             71666|
|    101|             50000|
|    105|             50000|
+-------+------------------+



In [0]:
#Find highest salary in each department
df_employee.groupBy("dept_id").agg(max("salary")).orderBy(col("dept_id").desc()).show()



+-------+-----------+
|dept_id|max(salary)|
+-------+-----------+
|    105|    50000.0|
|    104|    55000.0|
|    103|    45000.0|
|    102|    80000.0|
|    101|    50000.0|
+-------+-----------+

+-------+-------+
|dept_id|max_sal|
+-------+-------+
|    101|50000.0|
|    102|80000.0|
|    103|45000.0|
|    104|55000.0|
|    105|50000.0|
+-------+-------+



In [0]:
#Rank employees by salary within department
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank
df_employee.withColumn("rank",dense_rank().over(Window.partitionBy("dept_id").orderBy(col("salary").desc()))).select("emp_name","dept_id","salary","rank").show()


+--------+-------+-------+----+
|emp_name|dept_id| salary|rank|
+--------+-------+-------+----+
|    Ravi|    101|50000.0|   1|
|   Arjun|    102|80000.0|   1|
|   Sneha|    102|70000.0|   2|
|    Amit|    102|65000.0|   3|
|   Kiran|    103|45000.0|   1|
|    Neha|    104|55000.0|   1|
|  Aarush|    105|50000.0|   1|
+--------+-------+-------+----+



In [0]:
#Find highest paid employee in each department
df_employee.withColumn("rank",dense_rank().over(Window.partitionBy("dept_id").orderBy(col("salary").desc()))).filter(col("rank")==1).select("emp_name","dept_id","salary").show()

+--------+-------+-------+
|emp_name|dept_id| salary|
+--------+-------+-------+
|    Ravi|    101|50000.0|
|   Arjun|    102|80000.0|
|   Kiran|    103|45000.0|
|    Neha|    104|55000.0|
|  Aarush|    105|50000.0|
+--------+-------+-------+



In [0]:
#Running total of salaries department-wise
df_employee.withColumn("running",sum(col("salary")).over(Window.partitionBy("dept_id").orderBy(col("salary").desc()))).select("emp_name","dept_id","salary","running").show()

+--------+-------+-------+--------+
|emp_name|dept_id| salary| running|
+--------+-------+-------+--------+
|    Ravi|    101|50000.0| 50000.0|
|   Arjun|    102|80000.0| 80000.0|
|   Sneha|    102|70000.0|150000.0|
|    Amit|    102|65000.0|215000.0|
|   Kiran|    103|45000.0| 45000.0|
|    Neha|    104|55000.0| 55000.0|
|  Aarush|    105|50000.0| 50000.0|
+--------+-------+-------+--------+



In [0]:
df_customer=spark.table("customer")
df_order=spark.table("orders")   
df_invoice=spark.table("invoice")

In [0]:
#Get customer name with order details
from pyspark.sql.functions import col,row_number,dense_rank,rank,avg,floor,max,sum
from pyspark.sql.window import Window

df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner").select("c.cust_name","o.order_id","o.order_date","o.order_amount").show()


+---------+--------+----------+------------+
|cust_name|order_id|order_date|order_amount|
+---------+--------+----------+------------+
|     Ravi|     111|2026-01-02|      2500.0|
|      Sam|     112|2026-01-03|      7200.0|
|     John|     113|2026-01-03|      1800.0|
|     John|     114|2026-01-05|      9500.0|
|     Mike|     115|2026-01-05|      4200.0|
|    David|     116|2026-01-06|      6100.0|
|    David|     117|2026-01-07|      3300.0|
|     Sara|     118|2026-01-08|      8700.0|
|     Sara|     119|2026-01-08|      2600.0|
|     Ravi|     120|2026-01-09|      5100.0|
|     Ravi|     121|2026-01-10|      7800.0|
|      Sam|     122|2026-01-10|      2900.0|
|      Sam|     123|2026-01-11|      6400.0|
|     Ravi|     124|2026-01-12|      1200.0|
|    David|     125|2026-01-12|      9900.0|
|     Sara|     126|2026-01-13|      4500.0|
|     Ravi|     127|2026-01-13|      7600.0|
|     Mike|     128|2026-01-14|      3400.0|
|     John|     129|2026-01-14|      8800.0|
|     Ravi

In [0]:
#Get customer orders with delivery date
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner").join(df_invoice.alias("i"),col("o.order_id")==col("i.order_id"),"inner").select("c.cust_name","o.order_id","o.order_date","o.order_amount","i.delivery_date").display()



cust_name,order_id,order_date,order_amount,delivery_date
Ravi,111,2026-01-02,2500.0,null
Sam,112,2026-01-03,7200.0,2026-01-06
John,113,2026-01-03,1800.0,2026-01-05
John,114,2026-01-05,9500.0,2026-01-09
Mike,115,2026-01-05,4200.0,2026-01-07
David,116,2026-01-06,6100.0,2026-01-10
David,117,2026-01-07,3300.0,2026-01-11
Sara,118,2026-01-08,8700.0,2026-01-13
Sara,119,2026-01-08,2600.0,2026-01-10
Ravi,120,2026-01-09,5100.0,2026-01-14


In [0]:
#Find customers who never placed orders
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"outer").filter(col("o.order_id").isNull()).select("c.cust_name").display()

df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"left_anti").select("c.cust_name").display()


cust_name
Anu
Chris
Tom
Jenny


cust_name
Tom
Jenny
Chris
Anu


In [0]:
#Find orders not yet delivered
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner").join(df_invoice.alias("i"),col("o.order_id")==col("i.order_id"),"inner").filter(col("i.delivery_date").isNull()).select("c.cust_name","o.order_id","o.order_date","o.order_amount","i.delivery_date").display()

cust_name,order_id,order_date,order_amount,delivery_date
Ravi,111,2026-01-02,2500.0,null


In [0]:
#Find total order amount by customer
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner").groupBy("c.cust_id","c.cust_name").agg(sum(col("o.order_amount")).alias("total_order_amount")).orderBy(col("c.cust_id").asc()).select("c.cust_id","c.cust_name","total_order_amount").display()

df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner") \
    .withColumn("total_order_amount",sum(col("o.order_amount")).over(Window.partitionBy("c.cust_id"))).select("c.cust_id","c.cust_name","total_order_amount").distinct().display()

cust_id,cust_name,total_order_amount
1,Ravi,25700.0
2,Sam,16500.0
3,John,20100.0
4,Mike,7600.0
5,David,19300.0
6,Sara,15800.0


cust_id,cust_name,total_order_amount
1,Ravi,25700.0
2,Sam,16500.0
3,John,20100.0
4,Mike,7600.0
5,David,19300.0
6,Sara,15800.0


In [0]:
#Find average order amount by each customer
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner").groupBy("c.cust_id","c.cust_name").agg(floor(avg(col("o.order_amount"))).alias("avg_order_amount")).orderBy(col("c.cust_id").asc()).select("c.cust_id","c.cust_name","avg_order_amount").display()


cust_id,cust_name,avg_order_amount
1,Ravi,4283
2,Sam,5500
3,John,6700
4,Mike,3800
5,David,6433
6,Sara,5266


In [0]:
#Find highest order amount customer-wise
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner").groupBy("c.cust_id","c.cust_name").agg(floor(max(col("o.order_amount"))).alias("highest_order_amount")).orderBy(col("c.cust_id").asc()).select("c.cust_id","c.cust_name","highest_order_amount").display()

df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner")\
    .withColumn("total_order_amount",max(col("o.order_amount")).over(Window.partitionBy("c.cust_id"))).select("c.cust_id","c.cust_name","total_order_amount").distinct().display()


cust_id,cust_name,highest_order_amount
1,Ravi,7800
2,Sam,7200
3,John,9500
4,Mike,4200
5,David,9900
6,Sara,8700


cust_id,cust_name,total_order_amount
1,Ravi,7800.0
2,Sam,7200.0
3,John,9500.0
4,Mike,4200.0
5,David,9900.0
6,Sara,8700.0


In [0]:
%sql
SELECT *
FROM (
    SELECT c.cust_name,
           SUM(o.order_amount) AS total_purchase,
           RANK() OVER(
               ORDER BY SUM(o.order_amount) DESC
           ) AS rnk
    FROM customer c
    JOIN orders o
    ON c.cust_id = o.cust_id
    GROUP BY c.cust_name
) t
WHERE rnk = 1;


cust_name,total_purchase,rnk
Ravi,25700.0,1


In [0]:
#1. Get employee name with department name
df_employee.alias("c").join(df_dept.alias("d"),col("c.dept_id")==col("d.dept_id"),"inner").select("c.emp_name","d.dept_name").show()

+--------+---------+
|emp_name|dept_name|
+--------+---------+
|    Ravi|       HR|
|    Amit|       IT|
|   Sneha|       IT|
|   Kiran|  Finance|
|    Neha|    Sales|
|   Arjun|       IT|
+--------+---------+



In [0]:
#Get all employees including those without departments
df_employee.alias("e").join(df_dept.alias("d"),col("e.dept_id")==col("d.dept_id"),"left").select("e.emp_name","d.dept_name").show()

+--------+---------+
|emp_name|dept_name|
+--------+---------+
|    Ravi|       HR|
|    Amit|       IT|
|   Sneha|       IT|
|   Kiran|  Finance|
|    Neha|    Sales|
|   Arjun|       IT|
|  Aarush|     NULL|
+--------+---------+



In [0]:
#Find total salary department-wise
df_dept.alias("d").join(df_employee.alias("e"),col("d.dept_id")==col("e.dept_id"),"inner").groupBy("d.dept_id").agg(sum(col("e.salary").alias("total_salary"))).show()

+-------+---------------------------+
|dept_id|sum(salary AS total_salary)|
+-------+---------------------------+
|    104|                    55000.0|
|    103|                    45000.0|
|    102|                   215000.0|
|    101|                    50000.0|
+-------+---------------------------+



In [0]:
# Rank employees by salary within department

window_spec=Window.partitionBy("dept_id").orderBy(col("salary").desc())

df_employee_ranked=df_employee.withColumn("rank",dense_rank().over(window_spec)).select("dept_id","rank")

df_employee_ranked.alias("e").join(df_dept.alias("d"),col("e.dept_id")==col("d.dept_id")).select("d.dept_name","e.rank").show()

df_employee.withColumn("rank",dense_rank().over(window_spec)).select("emp_id","dept_id","salary","rank").display()


+---------+----+
|dept_name|rank|
+---------+----+
|       HR|   1|
|       IT|   1|
|       IT|   2|
|       IT|   3|
|  Finance|   1|
|    Sales|   1|
+---------+----+



emp_id,dept_id,salary,rank
1,101,50000.0,1
6,102,80000.0,1
3,102,70000.0,2
2,102,65000.0,3
4,103,45000.0,1
5,104,55000.0,1
1,105,50000.0,1


In [0]:
#Find average salary by department
df_dept.alias("d").join(df_employee.alias("e"),col("d.dept_id")==col("e.dept_id"),"inner").groupBy("d.dept_id").agg(avg(col("e.salary").alias("total_salary"))).show()

+-------+---------------------------+
|dept_id|avg(salary AS total_salary)|
+-------+---------------------------+
|    104|                    55000.0|
|    103|                    45000.0|
|    102|          71666.66666666667|
|    101|                    50000.0|
+-------+---------------------------+



In [0]:
#Find highest salary in each department
df_dept.alias("d").join(df_employee.alias("e"),col("d.dept_id")==col("e.dept_id"),"inner").groupBy("d.dept_id").agg(max(col("e.salary"))).show()

+-------+-----------+
|dept_id|max(salary)|
+-------+-----------+
|    104|    55000.0|
|    103|    45000.0|
|    102|    80000.0|
|    101|    50000.0|
+-------+-----------+



In [0]:
# Find highest paid employee in each department
df_employee.withColumn("rank",row_number().over(Window.partitionBy("dept_id").orderBy(col("salary").desc()))).filter(col("rank")==1).select("emp_name","dept_id","salary").show()

+--------+-------+-------+
|emp_name|dept_id| salary|
+--------+-------+-------+
|    Ravi|    101|50000.0|
|   Arjun|    102|80000.0|
|   Kiran|    103|45000.0|
|    Neha|    104|55000.0|
|  Aarush|    105|50000.0|
+--------+-------+-------+



In [0]:
#Running total of salaries department-wise
df_employee.alias("e").join(df_dept.alias("d"),col("e.dept_id")==col("d.dept_id"),"inner").withColumn("running_sal",sum(col("salary")).over(
    Window.partitionBy("e.dept_id")
    .orderBy(col("e.salary").desc())
)).select("d.dept_id","d.dept_name","e.salary","running_sal").show()

+-------+---------+-------+-----------+
|dept_id|dept_name| salary|running_sal|
+-------+---------+-------+-----------+
|    101|       HR|50000.0|    50000.0|
|    102|       IT|80000.0|    80000.0|
|    102|       IT|70000.0|   150000.0|
|    102|       IT|65000.0|   215000.0|
|    103|  Finance|45000.0|    45000.0|
|    104|    Sales|55000.0|    55000.0|
+-------+---------+-------+-----------+



In [0]:
#Get employee and manager names
df_employee.alias("e").join(df_employee.alias("m"),col("e.manager_id")==col("m.emp_id"),"inner").select("e.emp_name","m.emp_name").show()

+--------+--------+
|emp_name|emp_name|
+--------+--------+
|   Kiran|    Ravi|
|    Neha|    Amit|
|    Amit|    Ravi|
|   Sneha|    Amit|
|   Kiran|  Aarush|
|    Amit|  Aarush|
+--------+--------+



In [0]:
#10. Find employees earning more than their managers
df_employee.alias("e").join(df_employee.alias("m"),col("e.manager_id")==col("m.emp_id"),"inner").filter(col("e.salary")>col("m.salary")).select("e.emp_name","e.salary","m.emp_name","m.salary").show()

df_employee.show()

+--------+-------+--------+-------+
|emp_name| salary|emp_name| salary|
+--------+-------+--------+-------+
|    Amit|65000.0|    Ravi|50000.0|
|   Sneha|70000.0|    Amit|65000.0|
|    Amit|65000.0|  Aarush|50000.0|
+--------+-------+--------+-------+

+------+--------+-------+----------+----------+-------+
|emp_id|emp_name| salary| hire_date|manager_id|dept_id|
+------+--------+-------+----------+----------+-------+
|     1|    Ravi|50000.0|2022-01-10|      NULL|    101|
|     2|    Amit|65000.0|2021-03-15|         1|    102|
|     3|   Sneha|70000.0|2020-07-20|         2|    102|
|     4|   Kiran|45000.0|2023-02-11|         1|    103|
|     5|    Neha|55000.0|2021-11-05|         2|    104|
|     6|   Arjun|80000.0|2019-08-18|      NULL|    102|
|     1|  Aarush|50000.0|2022-01-10|      NULL|    105|
+------+--------+-------+----------+----------+-------+



In [0]:
#Find employees with salary above company average
avg_sal=df_employee.select(floor(avg(col("salary")))).collect()[0][0]
display(avg_sal)
df_employee.select("emp_name","salary").filter(col("salary")>avg_sal).show()

59285

+--------+-------+
|emp_name| salary|
+--------+-------+
|    Amit|65000.0|
|   Sneha|70000.0|
|   Arjun|80000.0|
+--------+-------+



In [0]:

df_employee.withColumn(
    "avg_salary",
    avg("salary").over(Window.partitionBy())
).filter(
    col("salary") > col("avg_salary")
).select(
    "emp_name",
    "salary"
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------+-------+
|emp_name| salary|
+--------+-------+
|    Amit|65000.0|
|   Sneha|70000.0|
|   Arjun|80000.0|
+--------+-------+



In [0]:
#Find department with highest total salary
df_employee.groupBy("dept_id").agg(sum(col("salary")).alias("total_salary")).orderBy(col("total_salary").desc()).limit(1).show()

+-------+------------+
|dept_id|total_salary|
+-------+------------+
|    102|    215000.0|
+-------+------------+



In [0]:
#Find department with highest total salary
df_employee.groupBy("dept_id") \
    .agg(sum("salary").alias("total_salary")) \
    .withColumn(
        "rank",
        dense_rank().over(
            Window.orderBy(col("total_salary").desc())
        )
    ) \
    .filter(col("rank") == 1) \
    .show()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------+------------+----+
|dept_id|total_salary|rank|
+-------+------------+----+
|    102|    215000.0|   1|
+-------+------------+----+



In [0]:
#find duplicate salaries
df_employee.groupBy(
    "salary"
) \
.agg(count("*").alias("count")
) \
.filter(
col("count")<=1
)\
.show()





+-------+-----+
| salary|count|
+-------+-----+
|55000.0|    1|
|65000.0|    1|
|45000.0|    1|
|70000.0|    1|
|80000.0|    1|
+-------+-----+



In [0]:
#Find top 3 salaries in each department
df_employee.withColumn(
    "rank",
    dense_rank().over(
        Window.partitionBy("dept_id").orderBy(col("salary").desc())
    )
).filter(
    col("rank")<=3
)\
.select("dept_id","salary").show()


+-------+-------+
|dept_id| salary|
+-------+-------+
|    101|50000.0|
|    102|80000.0|
|    102|70000.0|
|    102|65000.0|
|    103|45000.0|
|    104|55000.0|
|    105|50000.0|
+-------+-------+



In [0]:
#write a query to filter duplicate records based on time stamp using windows functions
df_employee.show()

df_employee.withColumn(
    "rank",
    dense_rank().over(
        Window.partitionBy("emp_name") \
            .orderBy(col("hire_date").desc()
    )
)
).filter(col("rank")==1).select("emp_name","hire_date").show()


+------+--------+-------+----------+----------+-------+
|emp_id|emp_name| salary| hire_date|manager_id|dept_id|
+------+--------+-------+----------+----------+-------+
|     1|    Ravi|50000.0|2022-01-10|      NULL|    101|
|     2|    Amit|65000.0|2021-03-15|         1|    102|
|     3|   Sneha|70000.0|2020-07-20|         2|    102|
|     4|   Kiran|45000.0|2023-02-11|         1|    103|
|     5|    Neha|55000.0|2021-11-05|         2|    104|
|     6|   Arjun|80000.0|2019-08-18|      NULL|    102|
|     1|  Aarush|50000.0|2022-01-10|      NULL|    105|
+------+--------+-------+----------+----------+-------+

+--------+----------+
|emp_name| hire_date|
+--------+----------+
|    Ravi|2022-01-10|
|    Amit|2021-03-15|
|   Sneha|2020-07-20|
|   Kiran|2023-02-11|
|    Neha|2021-11-05|
|   Arjun|2019-08-18|
|  Aarush|2022-01-10|
+--------+----------+



In [0]:
%sql
select emp_id,count(hire_date) as count from employee group by emp_id  having count>1

emp_id,count
1,2


In [0]:
display(df_employee)

emp_id,emp_name,salary,hire_date,manager_id,dept_id
1,Ravi,50000.0,2022-01-10,null,101
2,Amit,65000.0,2021-03-15,1,102
3,Sneha,70000.0,2020-07-20,2,102
4,Kiran,45000.0,2023-02-11,1,103
5,Neha,55000.0,2021-11-05,2,104
6,Arjun,80000.0,2019-08-18,null,102
1,Aarush,50000.0,2022-01-10,null,105


## Aggregation Pattern

In [0]:
# find emp name,max salary in each department
df_employee.groupBy("dept_id").agg(
    max(col("salary")).alias("max_salary")) \
        .withColumnRenamed("dept_id","dept")\
        .join(df_employee.alias("e"),(col("e.dept_id")==col("dept"))&(col("e.salary")==col("max_salary")),"inner") \
        .select("e.emp_name","dept","e.salary")\
            .orderBy(col("dept"))\
                .show()


# using window function
df_employee.withColumn(
    "rank",
    dense_rank().over(
        Window.partitionBy("dept_id").orderBy(col("salary").desc())
    )
).filter(
    col("rank")==1
).select("dept_id","emp_name","salary")\
    .orderBy(col("dept_id"))\
      .show()


    

+--------+----+-------+
|emp_name|dept| salary|
+--------+----+-------+
|    Ravi| 101|50000.0|
|   Arjun| 102|80000.0|
|   Kiran| 103|45000.0|
|    Neha| 104|55000.0|
|  Aarush| 105|50000.0|
+--------+----+-------+

+-------+--------+-------+
|dept_id|emp_name| salary|
+-------+--------+-------+
|    101|    Ravi|50000.0|
|    102|   Arjun|80000.0|
|    103|   Kiran|45000.0|
|    104|    Neha|55000.0|
|    105|  Aarush|50000.0|
+-------+--------+-------+



In [0]:
# Find departments having more than 1 employee
df_employee.groupBy("dept_id").agg(count("*").alias("count")).filter(col("count")>1).show()

#using window function
df_employee.withColumn(
    "rank",
    count("*").over(
        Window.partitionBy("dept_id")
    )
).filter(
    col("rank")>1 \
    ).select("dept_id","rank").limit(1).show()



+-------+-----+
|dept_id|count|
+-------+-----+
|    102|    3|
+-------+-----+

+-------+----+
|dept_id|rank|
+-------+----+
|    102|   3|
+-------+----+



In [0]:
#Find average salary by department
df_employee.groupBy("dept_id").agg(floor(avg(col("salary"))).alias("avg_salary")).orderBy(col("dept_id").desc()).show() 

#using window function
df_employee.withColumn("avg_salary",avg("salary").over(
    Window.partitionBy("dept_id")
)).select("dept_id","avg_salary").orderBy(col("dept_id").desc()).show()

+-------+----------+
|dept_id|avg_salary|
+-------+----------+
|    105|     50000|
|    104|     55000|
|    103|     45000|
|    102|     71666|
|    101|     50000|
+-------+----------+

+-------+-----------------+
|dept_id|       avg_salary|
+-------+-----------------+
|    105|          50000.0|
|    104|          55000.0|
|    103|          45000.0|
|    102|71666.66666666667|
|    102|71666.66666666667|
|    102|71666.66666666667|
|    101|          50000.0|
+-------+-----------------+



## 2. Top-N Pattern

In [0]:
# employee with highest salary
df_employee.orderBy(col("salary").desc()).limit(1).show()

+------+--------+-------+----------+----------+-------+
|emp_id|emp_name| salary| hire_date|manager_id|dept_id|
+------+--------+-------+----------+----------+-------+
|     6|   Arjun|80000.0|2019-08-18|      NULL|    102|
+------+--------+-------+----------+----------+-------+



In [0]:
#Second highest salary
df_employee.withColumn(
    "rank",
    dense_rank().over(
        Window.orderBy(col("salary").desc())
    )
).filter(
    col("rank")==2
).select("emp_name","dept_id","salary").show()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------+-------+-------+
|emp_name|dept_id| salary|
+--------+-------+-------+
|   Sneha|    102|70000.0|
+--------+-------+-------+



In [0]:
#Top 3 salaries
df_employee.withColumn(
    "rank",
    dense_rank().over(
        Window.orderBy(col("salary").desc())
    )
).filter(
    col("rank")<=3
).select("emp_name","dept_id","salary").show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------+-------+-------+
|emp_name|dept_id| salary|
+--------+-------+-------+
|   Arjun|    102|80000.0|
|   Sneha|    102|70000.0|
|    Amit|    102|65000.0|
+--------+-------+-------+



## 3. Department-wise Ranking

In [0]:
#Highest salary employee in each department
df_employee.withColumn(
    "rank",
    row_number().over(
        Window.partitionBy("dept_id").orderBy(col("salary").desc())
    )
).filter(
    col("rank")==1
).select("emp_name","dept_id","salary").show()

+--------+-------+-------+
|emp_name|dept_id| salary|
+--------+-------+-------+
|    Ravi|    101|50000.0|
|   Arjun|    102|80000.0|
|   Kiran|    103|45000.0|
|    Neha|    104|55000.0|
|  Aarush|    105|50000.0|
+--------+-------+-------+



## 4. Self Join Pattern

In [0]:
#Employees earning more than their manager
df_employee.alias("e").join(df_employee.alias("m"),col("e.manager_id")==col("m.emp_id"),"inner").filter(col("e.salary")>col("m.salary")).select("*").show()

+------+--------+-------+----------+----------+-------+------+--------+-------+----------+----------+-------+
|emp_id|emp_name| salary| hire_date|manager_id|dept_id|emp_id|emp_name| salary| hire_date|manager_id|dept_id|
+------+--------+-------+----------+----------+-------+------+--------+-------+----------+----------+-------+
|     2|    Amit|65000.0|2021-03-15|         1|    102|     1|    Ravi|50000.0|2022-01-10|      NULL|    101|
|     3|   Sneha|70000.0|2020-07-20|         2|    102|     2|    Amit|65000.0|2021-03-15|         1|    102|
|     2|    Amit|65000.0|2021-03-15|         1|    102|     1|  Aarush|50000.0|2022-01-10|      NULL|    105|
+------+--------+-------+----------+----------+-------+------+--------+-------+----------+----------+-------+



In [0]:
#Employee-manager pairs
df_employee.alias("e").join(df_employee.alias("m"),col("e.manager_id")==col("m.emp_id"),"inner").select("e.emp_name","m.emp_name").show()
df_employee.show()

+--------+--------+
|emp_name|emp_name|
+--------+--------+
|   Kiran|    Ravi|
|    Neha|    Amit|
|    Amit|    Ravi|
|   Sneha|    Amit|
|   Kiran|  Aarush|
|    Amit|  Aarush|
+--------+--------+

+------+--------+-------+----------+----------+-------+
|emp_id|emp_name| salary| hire_date|manager_id|dept_id|
+------+--------+-------+----------+----------+-------+
|     1|    Ravi|50000.0|2022-01-10|      NULL|    101|
|     2|    Amit|65000.0|2021-03-15|         1|    102|
|     3|   Sneha|70000.0|2020-07-20|         2|    102|
|     4|   Kiran|45000.0|2023-02-11|         1|    103|
|     5|    Neha|55000.0|2021-11-05|         2|    104|
|     6|   Arjun|80000.0|2019-08-18|      NULL|    102|
|     1|  Aarush|50000.0|2022-01-10|      NULL|    105|
+------+--------+-------+----------+----------+-------+



## 5. Correlated Subquery Pattern

In [0]:
#Employees earning above department average
df_employee.withColumn("dept_avg",avg(
    "salary").over(Window.partitionBy("dept_id")))\
        .filter(col("salary")>col("dept_avg")).select("emp_name","dept_id","salary").show()


+--------+-------+-------+
|emp_name|dept_id| salary|
+--------+-------+-------+
|   Arjun|    102|80000.0|
+--------+-------+-------+



In [0]:
#Employees earning maximum salary in department
df_employee.withColumn("maxsal",max(
    "salary").over(Window.partitionBy("dept_id")))\
        .filter(col("salary")==col("maxsal")).select("emp_name","dept_id","salary").show()

df_employee.withColumn("rank",dense_rank().over(Window.partitionBy("dept_id").orderBy(col("salary").desc()))).filter(col("rank")==1).select("emp_name","dept_id","salary").show()

+--------+-------+-------+
|emp_name|dept_id| salary|
+--------+-------+-------+
|    Ravi|    101|50000.0|
|   Kiran|    103|45000.0|
|    Neha|    104|55000.0|
|   Arjun|    102|80000.0|
|  Aarush|    105|50000.0|
+--------+-------+-------+

+--------+-------+-------+
|emp_name|dept_id| salary|
+--------+-------+-------+
|    Ravi|    101|50000.0|
|   Arjun|    102|80000.0|
|   Kiran|    103|45000.0|
|    Neha|    104|55000.0|
|  Aarush|    105|50000.0|
+--------+-------+-------+



6. Duplicate Detection Pattern

In [0]:
#Employees having same salary
df_employee.withColumn("cnt",count("*").over(Window.partitionBy("salary"))).filter(col("cnt")>1).select("emp_name","salary").show()



+--------+-------+
|emp_name| salary|
+--------+-------+
|    Ravi|50000.0|
|  Aarush|50000.0|
+--------+-------+



/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------+-------+
|emp_name| salary|
+--------+-------+
|    Ravi|50000.0|
|  Aarush|50000.0|
|    Neha|55000.0|
|    Amit|65000.0|
|   Sneha|70000.0|
|   Arjun|80000.0|
+--------+-------+



## Relative Ranking Pattern

In [0]:
#name of employee whose sal greater than two other employee from same dept

from pyspark.sql.functions import col, max,count,dense_rank
from pyspark.sql.window import Window
result = (
    df_employee.groupBy("dept_id")
      .agg(
          count("*").alias("emp_cnt"),
          max("salary").alias("max_salary")
      )
       .filter(col("emp_cnt") > 2)
       .withColumnRenamed("dept_id","dept_no")
      .join(
          df_employee.alias("e1"),
          (col("e1.dept_id") == col("dept_no")) &
          (col("e1.salary") == col("max_salary"))
      )
      .select("e1.emp_name", "e1.dept_id", "e1.salary")
)

result.show()

# using window functions
w_dept = Window.partitionBy("dept_id")
w_rank = Window.partitionBy("dept_id").orderBy(col("salary").desc())

result = (df_employee
          .withColumn("emp_cnt", count("*").over(w_dept))
          .withColumn("rnk", dense_rank().over(w_rank))
            .filter((col("emp_cnt") > 2) & (col("rnk") == 1))
           .select("emp_name", "dept_id", "salary"))

result.show()





+--------+-------+-------+
|emp_name|dept_id| salary|
+--------+-------+-------+
|   Arjun|    102|80000.0|
+--------+-------+-------+

+--------+-------+-------+
|emp_name|dept_id| salary|
+--------+-------+-------+
|   Arjun|    102|80000.0|
+--------+-------+-------+

